<a href="https://colab.research.google.com/github/sreent/machine-learning/blob/main/Training%20and%20Tuning/13%20Exercise%3A%20Putting%20It%20All%20Together.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Diabetes Case Study

You now have had the opportunity to work with a range of supervised machine learning techniques for both classification and regression.  Before you apply these in the project, let's do one more example to see how the machine learning process works from beginning to end with another popular dataset.

We will start out by reading in the dataset and our necessary libraries.  You will then gain an understanding of how to optimize a number of models using grid searching as you work through the notebook.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import MultinomialNB
import matplotlib.pyplot as plt
from sklearn.svm import SVC
import seaborn as sns


def check_one(answers_one):
    '''
    INPUT:
    answers_one - a dictionary with key-value pairs associated with question 1

    OUTPUT:
    nothing returned
    print a statement related to the correctness of the answers
    '''
    a = '0.65'
    b = '0'
    c = 'Age'
    d = '0.35'
    e = 'Glucose'
    f = '0.5'
    g = "More than zero"

    answers_one_1 = {
        'The proportion of diabetes outcomes in the dataset': d,
        'The number of missing data points in the dataset': b,
        'A dataset with a symmetric distribution': e,
        'A dataset with a right-skewed distribution': c,
        'This variable has the strongest correlation with the outcome': e
    }

    if answers_one == answers_one_1:
        print("Awesome! These all look great!")

    if answers_one['The proportion of diabetes outcomes in the dataset'] != answers_one_1['The proportion of diabetes outcomes in the dataset']:
        print("Oops!  That doesn't look like the proportion of 1's in the outcomes column.  I saw closer to 35% using the describe() method.")

    if answers_one['The number of missing data points in the dataset'] != answers_one_1['The number of missing data points in the dataset']:
        print("Oops!  That doesn't look like the right number of missing values.  I actually couldn't find any missing values")

    if answers_one['A dataset with a symmetric distribution'] != answers_one_1['A dataset with a symmetric distribution']:
        print("Oops!  Of the two columns above, Glucose is actually the symmetric column.  You can see this by running the .hist() method on your dataframe.")

    if answers_one['A dataset with a right-skewed distribution'] != answers_one_1['A dataset with a right-skewed distribution']:
        print("Oops!  Of the two columns above, Age is actually the right-skewed column.  You can see this by running the .hist() method on your dataframe.")

    if answers_one['This variable has the strongest correlation with the outcome'] != answers_one_1['This variable has the strongest correlation with the outcome']:
        print("Oops!  Besides Outcome itself, the column that is most correlated with the Outcome variable is actually Glucose.")


def print_metrics(y_true, preds, model_name=None):
    '''
    INPUT:
    y_true - the y values that are actually true in the dataset (numpy array or pandas series)
    preds - the predictions for those values from some model (numpy array or pandas series)
    model_name - (str - optional) a name associated with the model if you would like to add it to the print statements

    OUTPUT:
    None - prints the accuracy, precision, recall, and F1 score
    '''
    if model_name == None:
        print('Accuracy score: ', format(accuracy_score(y_true, preds)))
        print('Precision score: ', format(precision_score(y_true, preds)))
        print('Recall score: ', format(recall_score(y_true, preds)))
        print('F1 score: ', format(f1_score(y_true, preds)))
        print('\n\n')

    else:
        print('Accuracy score for ' + model_name + ' :' , format(accuracy_score(y_true, preds)))
        print('Precision score ' + model_name + ' :', format(precision_score(y_true, preds)))
        print('Recall score ' + model_name + ' :', format(recall_score(y_true, preds)))
        print('F1 score ' + model_name + ' :', format(f1_score(y_true, preds)))
        print('\n\n')


def check_best(best_model):
    '''
    INPUT:
    best_model - a string of the best model

    OUTPUT:
    print a statement related to if the best model matches what the solution found
    '''
    a = 'randomforest'
    b = 'adaboost'
    c = 'supportvector'

    if best_model == b:
        print("Nice!  It looks like your best model matches the best model I found as well!  It makes sense to use f1 score to determine best in this case given the imbalance of classes.  There might be justification for precision or recall being the best metric to use as well - precision showed to be best with adaboost again.  With recall, SVMs proved to be the best for our models.")

    else:
        print("That wasn't the model I had in mind... It makes sense to use f1 score to determine best in this case given the imbalance of classes.  There could also be justification for precision or recall being the best metric to use as well - precision showed to be best with adaboost.  With recall, SVMs proved to be the best for our models.")


def check_q_seven(sol_seven):
    '''
    INPUT:
    solution dictionary for part seven
    OUTPUT:
    prints statement related to correctness of dictionary
    '''
    a = 'Age'
    b = 'BloodPressure'
    c = 'BMI'
    d = 'DiabetesPedigreeFunction'
    e = 'Insulin'
    f = 'Glucose'
    g = 'Pregnancy'
    h = 'SkinThickness'



    sol_seven_1 = {
        'The variable that is most related to the outcome of diabetes' : f,
        'The second most related variable to the outcome of diabetes' : c,
        'The third most related variable to the outcome of diabetes' : a,
        'The fourth most related variable to the outcome of diabetes' : d
    }

    if sol_seven == sol_seven_1:
        print("That's right!  Some of these were expected, but some were a bit unexpected too!")

    else:
        print("That doesn't look like what I expected, but maybe your feature importances were different - that can definitely happen.  Take a look at the best_estimator_.feature_importances_ portion of your fitted model.")



In [ ]:
# Import our libraries
import pandas as pd
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
import matplotlib.pyplot as plt
from sklearn.svm import SVC
import seaborn as sns
sns.set(style="ticks")

%matplotlib inline

# Read in our dataset
# URL for our dataset, diabetes.csv
URL = "https://drive.google.com/file/d/1C7320OhkSBGVFw6UdL8oh_92Lr1O4u1T/view?usp=sharing"
FILE_PATH = "https://drive.google.com/uc?export=download&id=" + URL.split("/")[-2]

diabetes = pd.read_csv(FILE_PATH)

# Take a look at the first few rows of the dataset
diabetes.head()

Because this course has been aimed at understanding machine learning techniques, we have largely ignored items related to parts of the data analysis process that come before building machine learning models - exploratory data analysis, feature engineering, data cleaning, and data wrangling.  

> **Step 1:** Let's do a few steps here.  Take a look at some of usual summary statistics calculated to accurately match the values to the appropriate key in the dictionary below.

In [ ]:
# Cells for work


In [ ]:
# Possible keys for the dictionary
a = '0.65'
b = '0'
c = 'Age'
d = '0.35'
e = 'Glucose'
f = '0.5'
g = "More than zero"

# Fill in the dictionary with the correct values here
# Note: A right-skewed distribution is one in which most values are clustered around the left tail while the right tail  is longer.
answers_one = {
    'The proportion of diabetes outcomes in the dataset': # add letter here,
    'The number of missing data points in the dataset': # add letter here,
    'A dataset with a symmetric distribution': # add letter here,
    'A dataset with a right-skewed distribution': # add letter here,
    'This variable has the strongest correlation with the outcome': # add letter here
}

# Just to check your answer, don't change this
check_one(answers_one)

> **Step 2**: Since our dataset here is quite clean, we will jump straight into the machine learning.  Our goal here is to be able to predict cases of diabetes.  First, you need to identify the y vector and X matrix.  Then, the following code will divide your dataset into training and test data.   

In [ ]:
y = # Pull y column
X = # Pull X variable columns

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Now that you have a training and testing dataset, we need to create some models that and ultimately find the best of them.  However, unlike in earlier lessons, where we used the defaults, we can now tune these models to be the very best models they can be.

It can often be difficult (and extremely time consuming) to test all the possible hyperparameter combinations to find the best models.  Therefore, it is often useful to set up a randomized search.  

In practice, randomized searches across hyperparameters have shown to be more time confusing, while still optimizing quite well.  One article related to this topic is available [here](https://blog.h2o.ai/2016/06/hyperparameter-optimization-in-h2o-grid-search-random-search-and-the-future/).  The documentation for using randomized search in sklearn can be found [here](http://scikit-learn.org/stable/auto_examples/model_selection/plot_randomized_search.html#sphx-glr-auto-examples-model-selection-plot-randomized-search-py) and [here](http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html).

In order to use the randomized search effectively, you will want to have a pretty reasonable understanding of the distributions that best give a sense of your hyperparameters.  Understanding what values are possible for your hyperparameters will allow you to write a grid search that performs well (and doesn't break).

> **Step 3**: In this step, I will show you how to use randomized search, and then you can set up grid searches for the other models in Step 4.  However, you will be helping, as I don't remember exactly what each of the hyperparameters in SVMs do.  Match each hyperparameter to its corresponding tuning functionality.



In [ ]:
# build a classifier
clf_rf = RandomForestClassifier()

# Set up the hyperparameter search
param_dist = {"max_depth": [3, None],
              "n_estimators": list(range(10, 200)),
              "max_features": list(range(1, X_test.shape[1]+1)),
              "min_samples_split": list(range(2, 11)),
              "min_samples_leaf": list(range(1, 11)),
              "bootstrap": [True, False],
              "criterion": ["gini", "entropy"]}


# Run a randomized search over the hyperparameters
random_search = RandomizedSearchCV(clf_rf, param_distributions=param_dist)

# Fit the model on the training data
random_search.fit(X_train, y_train)

# Make predictions on the test data
rf_preds = random_search.best_estimator_.predict(X_test)

print_metrics(y_test, rf_preds, 'random forest')

> **Step 4**: Now that you have seen how to run a randomized grid search using random forest, try this out for the AdaBoost and SVC classifiers.  You might also decide to try out other classifiers that you saw earlier in the lesson to see what works best.

In [ ]:
# build a classifier for ada boost


# Set up the hyperparameter search
# look at  setting up your search for n_estimators, learning_rate
# http://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html


# Run a randomized search over the hyperparameters


# Fit the model on the training data


# Make predictions on the test data
ada_preds =

# Return your metrics on test data
print_metrics(y_test, ada_preds, 'adaboost')

In [ ]:
# build a classifier for support vector machines


# Set up the hyperparameter search
# look at setting up your search for C (recommend 0-10 range),
# kernel, and degree
# http://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html



# Run a randomized search over the hyperparameters


# Fit the model on the training data


# Make predictions on the test data
svc_preds =


# Return your metrics on test data
print_metrics(y_test, svc_preds, 'svc')

> **Step 5**: Use the test below to see if your best model matched, what we found after running the grid search.  

In [ ]:
a = 'randomforest'
b = 'adaboost'
c = 'supportvector'

best_model =  # put your best model here as a string or variable

# See if your best model was also mine.
# Notice these might not match depending your search!
check_best(best_model)

Once you have found your best model, it is also important to understand why it is performing well.  In regression models where you can see the weights, it can be much easier to interpret results.

> **Step 6**:  Despite the fact that your models here are more difficult to interpret, there are some ways to get an idea of which features are important.  Using the "best model" from the previous question, find the features that were most important in helping determine if an individual would have diabetes or not. Do your conclusions match what you might have expected during the exploratory phase of this notebook?

In [ ]:
# Show your work here - the plot below was helpful for me
# https://stackoverflow.com/questions/44101458/random-forest-feature-importance-chart-using-python


> **Step 7**:  Using your results above to complete the dictionary below.

In [ ]:
# Check your solution by matching the correct values in the dictionary
# and running this cell
a = 'Age'
b = 'BloodPressure'
c = 'BMI'
d = 'DiabetesPedigreeFunction'
e = 'Insulin'
f = 'Glucose'
g = 'Pregnancy'
h = 'SkinThickness'



sol_seven = {
    'The variable that is most related to the outcome of diabetes' : # letter here,
    'The second most related variable to the outcome of diabetes' : # letter here,
    'The third most related variable to the outcome of diabetes' : # letter here,
    'The fourth most related variable to the outcome of diabetes' : # letter here
}

check_q_seven(sol_seven)

> **Step 8**:  Now provide a summary of what you did through this notebook, and how you might explain the results to a non-technical individual.  When you are done, check out the solution notebook by clicking the orange icon in the upper left.